# Preprocessing Prototype

**Project:** Power Consumption MLOps (Tetouan City)  
**Goal:** Prototype the preprocessing logic that will be extracted into `DataProcessor` in Phase 1.  
**Output:** Working preprocessing code in this notebook → extract to `src/tetouan_power/data_processor.py`.

> **Workflow:** Phase 0 (EDA in `00_initial_eda.ipynb`) → This notebook (prototype) → Phase 1 (extract to `src/`).  
> Do not write `DataProcessor` from scratch in `src/`. First validate the logic here.

## 1) Setup & Load Config

In [ ]:
from pathlib import Path

import pandas as pd

from tetouan_power.config import ProjectConfig

# Paths (same as EDA notebook)
REPO_ROOT = Path.cwd().parents[0]
DATA_PATH = REPO_ROOT / "data" / "raw" / "tetouan-power-consumption.csv"
CONFIG_PATH = REPO_ROOT / "project_config.yaml"

assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"
config = ProjectConfig.from_yaml(str(CONFIG_PATH), env="dev")
pd.set_option("display.max_columns", 50)

## 2) Load Raw Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head(3)

## 3) Column Mapping (Raw → snake_case)

Tetouan CSV has raw column names. We map them to snake_case for consistency.

| Raw Column | Renamed |
|------------|--------|
| DateTime | datetime |
| Temperature | temperature |
| Humidity | humidity |
| Wind Speed | wind_speed |
| general diffuse flows | general_diffuse_flows |
| diffuse flows | diffuse_flows |
| Zone 1 Power Consumption | zone1_consumption |
| Zone 2  Power Consumption | zone2_consumption |
| Zone 3  Power Consumption | zone3_consumption |

> Note: Zone 2 and Zone 3 have a double space in the raw name.

In [ ]:
COLUMN_RENAME = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_consumption",
    "Zone 2  Power Consumption": "zone2_consumption",
    "Zone 3  Power Consumption": "zone3_consumption",
}

df = df.rename(columns=COLUMN_RENAME)
df.columns.tolist()

## 4) Parse DateTime

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
df.info()

## 5) Temporal Features

Extract hour, day_of_week, month, is_weekend from datetime. These are **mandatory** for time series — EDA showed strong daily and seasonal patterns. They are in `project_config.yaml` `num_features`.

For time series, it is also done lags, but for this project im going to skip it.

In [ ]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

## 6) Handle Missing Values

Tetouan dataset is mostly clean (no nulls in EDA). For robustness, we fill or drop if any appear.

In [ ]:
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print("Nulls found:", null_counts[null_counts > 0].to_dict())
    df = df.dropna()  # or fill with median/mean for numeric columns
else:
    print("No nulls found.")

## 7) Select Columns & Generate id

Keep: `datetime` + `num_features` + `cat_features` + `[target]` + `["id"]`.  
**Keep `datetime`** — needed for time-based split and traceability.  
**`id`** — unique row identifier from datetime (for Delta tables, MLflow lineage). Use `datetime.astype(str)` so each row has a stable, interpretable id.

In [ ]:
relevant_columns = ["datetime"] + config.num_features + config.cat_features + [config.target]
df = df[relevant_columns].copy()
df["id"] = df["datetime"].astype(str)

print("Final columns:", df.columns.tolist())
df.head(3)

## 8) Sanity Check

Verifies preprocessing produced the expected output before moving on:
- **Columns** — exactly `datetime` + config columns + `id`
- **id dtype** — string (object), not numeric
- **No nulls** — all rows are complete

In [ ]:
expected_columns = set(relevant_columns + ["id"])
assert set(df.columns) == expected_columns, f"Expected {expected_columns}, got {set(df.columns)}"
assert pd.api.types.is_string_dtype(df["id"]), "id must be string type"
assert df.isnull().sum().sum() == 0, "No nulls allowed"
print("✅ Preprocessing prototype complete.")

## 9) Time-Based Split

**No random split.** Tetouan is time series — we must preserve temporal order.  
From `docs/00-problem-statement-v1.md`:
- **Train:** 2017-01 → 2017-09
- **Val:** 2017-10 → 2017-11
- **Test:** 2017-12 → 2017-12-30

Random split would leak future data into training.

In [ ]:
train_end = config.split.train_end if config.split else "2017-10-01"
val_end = config.split.val_end if config.split else "2017-12-01"

train_set = df[df["datetime"] < train_end]
val_set = df[(df["datetime"] >= train_end) & (df["datetime"] < val_end)]
test_set = df[df["datetime"] >= val_end]

print(f"Train: {len(train_set)} (Jan–Sep)")
print(f"Val:   {len(val_set)} (Oct–Nov)")
print(f"Test:  {len(test_set)} (Dec)")

print("\n")

print("Train Data")
print(f"- Min. Date: {train_set['datetime'].min()}")
print(f"- Max. Date: {train_set['datetime'].max()}")

print("\n")

print("Validation Data")
print(f"- Min. Date: {val_set['datetime'].min()}")
print(f"- Max. Date: {val_set['datetime'].max()}")

print("\n")

print("Test Data")
print(f"- Min. Date: {test_set['datetime'].min()}")
print(f"- Max. Date: {test_set['datetime'].max()}")

---

**Next:** Extract this logic into `src/tetouan_power/data_processor.py` (Step 6b in Phase 1).  
`DataProcessor.split_data()` should use this time-based split, not random `train_test_split`.

# 10)  Generating synthetic data

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
rows = []
date_ranges = [
    # (start, end, count) — spread across the 3 split periods
    ("2017-01-15", "2017-09-15", 8),  # train period
    ("2017-10-10", "2017-11-20", 6),  # validation period
    ("2017-12-05", "2017-12-25", 6),  # test period
]

for start, end, n in date_ranges:
    timestamps = pd.date_range(start, end, periods=n)
    for ts in timestamps:
        rows.append(
            {
                "DateTime": ts.strftime("%#m/%#d/%Y %H:%M"),  # Windows: %#m/%#d  Unix: %-m/%-d
                "Temperature": round(rng.uniform(5, 35), 3),
                "Humidity": round(rng.uniform(30, 90), 1),
                "Wind Speed": round(rng.uniform(0, 8), 3),
                "general diffuse flows": round(rng.uniform(0.01, 0.6), 3),
                "diffuse flows": round(rng.uniform(0.01, 0.5), 3),
                "Zone 1 Power Consumption": round(rng.uniform(20000, 50000), 4),
                "Zone 2  Power Consumption": round(rng.uniform(15000, 35000), 4),
                "Zone 3  Power Consumption": round(rng.uniform(18000, 40000), 4),
            }
        )

df = pd.DataFrame(rows)

# Inject a few nulls so test_missing_value_handling exercises the dropna() path
df.loc[2, "Temperature"] = None
df.loc[5, "Humidity"] = None

df.to_csv("../tests/test_data/sample.csv", index=False)
print(f"Created {len(df)} rows ({df.isnull().sum().sum()} nulls injected)")
print(df.head(8))